In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

from langgraph.graph import (
    StateGraph,
    START,
    END
)

# ==========================================================
# CREATE VECTOR DATABASE
# ==========================================================

loader = TextLoader("data.txt")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(
    chunks,
    embedding
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

# ==========================================================
# STATE
# ==========================================================

class AgentState(TypedDict):

    question: str

    documents: list[Document]

    answer: str


# ==========================================================
# LLM
# ==========================================================

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# ==========================================================
# RETRIEVE NODE
# ==========================================================

def retrieve(state: AgentState):

    docs = retriever.invoke(
        state["question"]
    )

    return {
        "documents": docs
    }

# ==========================================================
# GENERATE NODE
# ==========================================================

def generate(state: AgentState):

    context = "\n\n".join(
        doc.page_content
        for doc in state["documents"]
    )

    prompt = f"""
Answer the question only from the provided context.

Context:
{context}

Question:
{state["question"]}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

# ==========================================================
# GRAPH
# ==========================================================

builder = StateGraph(AgentState)

builder.add_node(
    "retrieve",
    retrieve
)

builder.add_node(
    "generate",
    generate
)

builder.add_edge(
    START,
    "retrieve"
)

builder.add_edge(
    "retrieve",
    "generate"
)

builder.add_edge(
    "generate",
    END
)

graph = builder.compile()

# ==========================================================
# RUN
# ==========================================================

result = graph.invoke(
    {
        "question":"What is LangGraph?"
    }
)

print(result["answer"])